# El objetivo de este notebook

Agruparemos reservas por tipología de habitación, buscaremos el BAR (Best Average Rate), y obtendremos el mejor precio obtenido para cada tipología con el objetivo de encontrar dos datos importantes, el BAR y el CPor.

---

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
import sys
# Agregamos el directorio padre (la raíz del proyecto) al path de Python
sys.path.append(os.path.abspath(os.path.join("..")))
# importamos la funcion directamente desde src
from src.load import load_to_sqlite

In [4]:
# Extraemos nuestro historico de reservas de la tabla 'clean_bookings'en la base de datos '\data\hotel_data.db'

def extract_from_sqlite(db_path: str = os.path.join("../data", "hotel_data.db"))  -> pd.DataFrame:
    """
    Conecta a la base de datos SQLite y extrae los datos de la tabla 'clean_bookings'.

    Args:
        db_path (str): Ruta al archivo de la base de datos SQLite.

    Returns:
        pd.DataFrame: DataFrame que contiene los datos extraídos.
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"❌ Error: No se encontró el archivo de base de datos en {db_path}. Asegúrate de que la base de datos exista.")
    
    print(f"⏳ Conectando a la base de datos en {db_path}...")

    # Crear la conexión a la base de datos SQLite
    engine = create_engine(f'sqlite:///{db_path}')

    # Consulta SQL para extraer los datos de la tabla 'clean_bookings'
    query = "SELECT * FROM clean_bookings"

    print("⏳ Ejecutando la consulta SQL para extraer los datos...")

    # Leer los datos de la tabla especificada en un DataFrame
    df = pd.read_sql_query(query, con=engine)

    print(f"✅ Extracción exitosa. Filas extraídas: {df.shape[0]}, Columnas: {df.shape[1]}")
    
    return df

In [5]:
df = extract_from_sqlite()

⏳ Conectando a la base de datos en ../data\hotel_data.db...
⏳ Ejecutando la consulta SQL para extraer los datos...
✅ Extracción exitosa. Filas extraídas: 119317, Columnas: 43


## 1. Agrupación por tipología de habitación

Nuestro dataset tiene una fila por reserva. Desacartaremos las reservas canceladas y agruparemos cada reserva por la tipología asignada, obteniendo el `.max()` en la columna `adr`.


In [6]:
def calculate_bar_by_room_type(df: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    """
    Calcula el BAR (Best Available Rate) por tipología de habitación,
    usando el máximo histórico de 'adr' como proxy de tarifa de lista.

    Incluye un sanity check de gap, para detectar si el máximo es un outlier aislado.
    """
    df_validas = df[df['is_canceled'] == 0].copy()

    resultados = []
    for (hotel, room_type), grupo in df_validas.groupby(['hotel', 'assigned_room_type'], observed=True):
        top_valores = grupo['adr'].nlargest(top_n).tolist()
        bar = top_valores[0]
        segundo = top_valores[1] if len(top_valores) > 1 else bar
        gap_pct = ((bar - segundo) / bar * 100) if bar > 0 else 0

        resultados.append({
            'hotel': hotel,
            'room_type': room_type,
            'bar': round(bar, 2),
            'segundo_valor': round(segundo, 2),
            'gap_pct': round(gap_pct, 1),
            'adr_promedio_referencia': round(grupo['adr'].mean(), 2),  # solo referencia, no es el BAR
            'total_reservas': len(grupo)
        })

    return pd.DataFrame(resultados).sort_values(['hotel', 'room_type'], ascending=[True, True])

In [7]:
bar_by_roomtype = calculate_bar_by_room_type(df)

In [8]:
display(bar_by_roomtype)

,hotel,room_type,bar,segundo_valor,gap_pct,adr_promedio_referencia,total_reservas
0,City Hotel,A,300.00,259.00,13.7,97.29,30084
1,City Hotel,B,263.55,248.49,5.7,93.72,1499
2,City Hotel,C,213.00,199.00,6.6,99.83,146
3,City Hotel,D,375.50,365.00,2.8,115.87,10706
4,City Hotel,E,451.50,287.00,36.4,138.56,1628
5,City Hotel,F,349.63,336.00,3.9,171.04,1299
6,City Hotel,G,510.00,372.33,27.0,176.85,571
7,City Hotel,K,283.23,260.00,8.2,52.77,267
8,Resort Hotel,A,337.00,305.00,9.5,80.08,10993
9,Resort Hotel,B,276.00,235.00,14.9,102.29,150


## 2. Clasificacion por tipologia y calculo de CPoR

In [9]:
# Ratio de CPoR como % del ADR promedio, según categoría (supuesto de industria)
CPOR_RATIO_BY_CATEGORY = {
    "Premium": 0.18,
    "Superior": 0.22,
    "Standard": 0.27,
    "Economy": 0.33,
}

def _clasificar_categoria(df_hotel: pd.DataFrame) -> pd.DataFrame:
    """
    Clasifica cada tipología de un hotel en Premium/Superior/Standard/Economy
    según su posición en el ranking de BAR (cuartiles), dentro de ese mismo hotel.
    """
    df_hotel = df_hotel.sort_values("bar", ascending=False).reset_index(drop=True)
    n = len(df_hotel)

    categorias = []
    for i in range(n):
        pct = i / n
        if pct < 0.25:
            categorias.append("Premium")
        elif pct < 0.5:
            categorias.append("Superior")
        elif pct < 0.75:
            categorias.append("Standard")
        else:
            categorias.append("Economy")

    df_hotel["categoria"] = categorias
    return df_hotel

def calculate_cpor_by_category(bar_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el CPoR estimado y la categoría (Premium/Superior/Standard/Economy)
    directamente desde el output de calculate_bar_by_room_type().
    """
    df = pd.concat(
        [_clasificar_categoria(grupo.copy()) for _, grupo in bar_df.groupby("hotel", observed=True)],
        ignore_index=True
    )

    df["ratio_cpor"] = df["categoria"].map(CPOR_RATIO_BY_CATEGORY)
    df["cpor"] = (df["adr_promedio_referencia"] * df["ratio_cpor"]).round(2)

    return df[["hotel", "room_type", "cpor", "categoria"]].sort_values(
        ["hotel", "room_type"]
    ).reset_index(drop=True)

In [10]:
resultado = calculate_cpor_by_category(bar_by_roomtype)

In [11]:
display(resultado)

,hotel,room_type,cpor,categoria
0,City Hotel,A,26.27,Standard
1,City Hotel,B,30.93,Economy
2,City Hotel,C,32.94,Economy
3,City Hotel,D,25.49,Superior
4,City Hotel,E,24.94,Premium
5,City Hotel,F,37.63,Superior
6,City Hotel,G,31.83,Premium
7,City Hotel,K,14.25,Standard
8,Resort Hotel,A,21.62,Standard
9,Resort Hotel,B,33.76,Economy


In [12]:
load_to_sqlite(resultado, table_name="cpor", if_exists_strategy="replace")

💾 Iniciando proceso de carga de datos...
    ├─ Conectando a la base de datos en 'data/hotel_data.db'...
    ├─ Insertando 17 registros en la tabla 'cpor' (Estrategia: replace)...
    └─ Verificación de carga exitosa: 17 filas en la tabla 'cpor'.
✅ Proceso de carga completado con éxito.



In [13]:
load_to_sqlite(bar_by_roomtype, "bar_by_roomtype", "replace")

💾 Iniciando proceso de carga de datos...
    ├─ Conectando a la base de datos en 'data/hotel_data.db'...
    ├─ Insertando 17 registros en la tabla 'bar_by_roomtype' (Estrategia: replace)...
    └─ Verificación de carga exitosa: 17 filas en la tabla 'bar_by_roomtype'.
✅ Proceso de carga completado con éxito.

